## Data preparation

In [ ]:
import pandas as pd
import torchvision.transforms as T
import numpy as np
import random
import torch
import torch.nn as nn
import datetime
import time
import os
import json
from transformers import get_linear_schedule_with_warmup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True  # Enforce deterministic algorithms
        torch.backends.cudnn.benchmark = False     # Disable benchmark for reproducibility

    os.environ['PYTHONHASHSEED'] = str(seed)       # Seed Python hashing, which can affect ordering
set_seed(42)


LABEL_COLS = [
    "Vaccum Cleaning", "Mopping the Floor", "Carry Warm Food",
    "Carry Cold Food", "Carry Drinks", "Carry Small Objects",
    "Carry Large Objects", "Cleaning", "Starting a conversation"
]


import sys
sys.path.append('..')

from models.buffers import NaiveRehearsalBuffer
from data_processing.data_utils import get_transform, serialize_transform, get_domain_dataloaders, pool_domain_dataloaders, pool_domain_dataloaders, get_crossvalidation_domain_loaders, IMAGENET_NORM, get_domain_dataloaders_from_hdf5
from models.training_utils import heuristic_dualbranch_batch, unified_train_loop, archive_domain_models
from models.heuristicSplitModel import DualBranchModel, DualBranchModel_fusions

## CL Models

In [ ]:
# social x env :
# env blank room
# env label 
# custom normalisation

# attention mask layer instead of binary mask cutouts
# if I run GradCAM I would like to see focus on humans in one image and focus on env in the other. Why not use gradCAM directly? i.e. apply 1 complementary attention map to each image, prime it with binary masks but allow for learning. 

# global x local :
# context size 2x > 1.5x, 3x, 4x
# context resolution 128x128 > 256x256, 512x512
# custom normalisation

# fusions :
# concatenation
# multiplication element (hadamard)
# FiLM - env produces gamma beta vectors, that gate soc 
# addition - ("parallel residual")
# gating 

# complex residuals addition
# complex residuals hadamard

# auxilary losses:
# contrastive embedding learning
# distilation loss - new prediction same as old 
# reconstruction losses:
#     recover bounding boxes
#     reconstruct Image
#     predict agumentation that was applied - flip rotate etc.
# !! loss related to correlation

# Different loss:
# CCC or CCC+MSE

# Test CL capability - no buffer
# nomasking vs my model vs robot zoom

#Add depth, and coordinates channels

#Compare to SOTA continuous domain adaptation - THATS my baseline conceptually
# though if my simple replay is better...does my count as a novel improvement?
# why just social and label is still good:
# if we can just use social and label that savesmemory and computation (measure)
# robot closeup doesnt add anything, 
# full scene + social cutout or attention map would add more
# or robot location, or all actor locations
# how about full image + social+depth+coords or label + soc+depth+coord


#from davide
# 1. auxilary loss showing that its better to clasify actions as inapprorpaite than appropriate
# maybe a score category classification? with a bias towards lower scores as desired could be a loss? - higher penalty for mislabeling inapropriate scores as approrpiate, lower penalty for appropriate action being inappropriate
# or a valence score - appropriate vs inapropriate
# 2. train on raw scores, not averages
# 3. ?train on all robots


#baseline - full image vs domain adaptation
#if full image then need for extracting more information from image not spliting or duplicating it
#specialised fusion, more info:depth, coordinates
#loss - just rmse not good
#corelation calculations - over domains, or combined | over tasks or samples | pooled or averaged
#training on full dataset, not avg

# corelation: averages over averages
# baselines - for now leave it - lgr, lvm + sth 
# weighing vectors - fusion experiment 
# loss experiment 
# figure out research questions 


In [ ]:
pd.read_pickle("../data/mean_data_pepper_train.pkl")

In [ ]:
import torch
import torch.nn as nn
import torchmetrics.functional as FM

# Base weighting functions (user can supply their own)
def no_weighting(y_true, y_pred=None):
    return torch.ones_like(y_true)

def symmetric_extreme_weighting(y_true, y_pred=None, lambda_extreme=3.0, mean_center=0.5, mean_width=0.2):
    weights = torch.ones_like(y_true)
    far_from_mean = (torch.abs(y_true - mean_center) >= mean_width)
    weights[far_from_mean] = lambda_extreme
    return weights


def asymmetric_extreme_weighting(y_true, y_pred=None, lambda_1=5.0, lambda_2=2.0):
    # y_pred needed to be passed in this function to use prediction-based weights
    weights = torch.ones_like(y_true)
    if y_pred is None:
        return weights
    weights[(y_true < 0.4) & (y_pred > 0.6)] = lambda_1
    weights[(y_true >= 0.4) & (y_true <= 0.6) & (y_pred > 0.6)] = lambda_2
    return weights

# Loss class with weighting function parameter
class MSELossWeighted(nn.Module):
    def __init__(self, weighting_func=no_weighting):
        super().__init__()
        self.mse = nn.MSELoss(reduction='none')
        self.weighting_func = weighting_func
    
    # matches nn.MSELoss: only y_pred, y_true arguments
    def forward(self, y_pred, y_true):
        weights = self.weighting_func(y_true, y_pred)
        losses = self.mse(y_pred, y_true)
        weighted_loss = (weights * losses).mean()
        return weighted_loss

def pool_correlations(corr_tensor: torch.Tensor) -> torch.Tensor:
    x = corr_tensor.clamp(-0.999999, 0.999999)
    z = torch.atanh(x)
    z_mean = z.mean()
    pooled_corr = torch.tanh(z_mean)
    return pooled_corr

class CCCLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, y_pred, y_true):
        # Compute CCC per feature dimension (returns shape [features])
        corr_per_feature = FM.concordance_corrcoef(y_pred, y_true)
        
        # Pool correlation values into single scalar robust to averaging bias
        pooled_corr = pool_correlations(corr_per_feature)
        
        # CCC loss
        loss = 1 - pooled_corr
        return loss


class CompoundLossFixedAlpha(nn.Module):
    def __init__(self, alpha=0.4, weighting_func=no_weighting):
        super().__init__()
        self.alpha = alpha
        self.mse_loss = MSELossWeighted(weighting_func)
        self.ccc_loss = CCCLoss()
    
    # matches nn.MSELoss signature: y_pred, y_true only
    def forward(self, y_pred, y_true):
        mse = self.mse_loss(y_pred, y_true)
        ccc = self.ccc_loss(y_pred, y_true)
        return self.alpha * mse + (1 - self.alpha) * ccc

class CompoundLossLearnableAlpha(nn.Module):
    def __init__(self, weighting_func=no_weighting):
        super().__init__()
        self.mse_loss = MSELossWeighted(weighting_func)
        self.ccc_loss = CCCLoss()
        self.log_var_mse = nn.Parameter(torch.zeros(1))
        self.log_var_ccc = nn.Parameter(torch.zeros(1))
    
    # matches nn.MSELoss signature: y_pred, y_true only
    def forward(self, y_pred, y_true):
        mse = self.mse_loss(y_pred, y_true)
        ccc = self.ccc_loss(y_pred, y_true)
        precision_mse = torch.exp(-self.log_var_mse)
        precision_ccc = torch.exp(-self.log_var_ccc)
        loss = precision_mse * mse + precision_ccc * ccc + (self.log_var_mse + self.log_var_ccc)
        return loss


In [ ]:
# # 1) Baseline MSE + CCC without weighting
# loss_fn1 = CompoundLossFixedAlpha(alpha=0.4)

# # 2) MSE with symmetric mean-based weighting
# loss_fn2 = MSELossWeighted(weighting_func=symmetric_extreme_weighting)

# # 3) Compound MSE + CCC with asymmetric domain weighting
# loss_fn3 = CompoundLossFixedAlpha(alpha=0.4, weighting_func=asymmetric_extreme_weighting)

# # 4) Compound MSE + CCC with learnable alpha, no weighting initially
# loss_fn4 = CompoundLossLearnableAlpha().to(device)

# # 5) Compound MSE + CCC with learnable alpha and asymmetric weighting
# loss_fn5 = CompoundLossLearnableAlpha(weighting_func=asymmetric_extreme_weighting).to(device)

In [ ]:
buffer = NaiveRehearsalBuffer(buffer_size=0)
domains=['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']
domain_dataloaders = get_domain_dataloaders_from_hdf5(hdf5_path="../data/mean_data_pepper.hdf5", domains=domains, img_path_cols=['image_path_env_boxes_plus', 'image_path_soc'], num_workers=0, set_first_element_as_domain_label=False, pin_memory=True, persistent_workers=False)
domain_dataloaders = pool_domain_dataloaders(domain_dataloaders)
domains = ['joint']
for current_domain in domains:
    train_loader = buffer.get_loader_with_replay(current_domain, domain_dataloaders[current_domain]['train'])
    print(f"Buffer after load: {buffer.get_domain_distribution()}")
    total_samples=0
    for batch in train_loader:
        inputs, labels, domain_labels = batch
        batch_size = batch[-1].shape[0]
        print(f'{domain_labels=}')
    total_samples += batch_size
    print(f'{total_samples=}')
    buffer.update_buffer(current_domain, domain_dataloaders[current_domain]['train'].dataset)
    print(f"Buffer after update: {buffer.get_domain_distribution()}")    

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    

default_domains = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']

testing_scenarios = {
    'silh.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, default_domains),
    'closeup.base': ('base', 60, 60, None, None, "../data/robotfocus128_pepper_data.hdf5", [get_transform((144,256), IMAGENET_NORM), get_transform((128,128), IMAGENET_NORM)], ['image_path_scene', 'image_path_focus_x2'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, default_domains),
    'order_complexup.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['Hallway', 'SmallOffice', 'MeetingRoom', 'BigOffice-3', 'BigOffice-2', 'Home']),
    'order_complexdown.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['Home', 'BigOffice-2', 'BigOffice-3', 'MeetingRoom', 'SmallOffice', 'Hallway']),
    'order_contrA.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['BigOffice-3', 'SmallOffice', 'Home', 'Hallway', 'MeetingRoom', 'BigOffice-2']),
    'order_contrB.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['MeetingRoom', 'Home', 'Hallway', 'BigOffice-2', 'SmallOffice', 'BigOffice-3']),
}
    
for name, (ablation, epochs, buffer_size, train_val_path, test_path, hdf5_path, transform_list, img_path_cols, scene_as_label, loss_fn, setup, joint_training, use_buffer, domains) in testing_scenarios.items():
        train_df = pd.read_pickle(train_val_path) if train_val_path is not None else None
        test_df = pd.read_pickle(test_path) if test_path is not None else None
        main_hdf5_dataset_path = hdf5_path
        setup=setup #{'branch':'mobilenetv2'}
        seed=42
        set_seed(seed)
        grad_clipping=True
        batch=(32, 64, 64)
        buffer_size=buffer_size
        branch_norm=True
        earlystopping=True
        epochs=epochs
        joint_training=joint_training
        scene_as_label=scene_as_label
        pin_memory=True
        persistent_workers=False
        num_workers=0
        use_buffer=use_buffer


        for fold in range(0,5):
            print(f"\nTesting fold: {fold}")

            hdf5_dataset_path = main_hdf5_dataset_path[:-5] + f'_fold{fold}.hdf5'

            if hdf5_path:
                domain_dataloaders = get_domain_dataloaders_from_hdf5(hdf5_path=hdf5_dataset_path, domains=domains, img_path_cols=img_path_cols, num_workers=num_workers, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            else:
                domain_dataloaders = get_domain_dataloaders(train_df, seed=seed, batch_sizes=batch, img_path_cols=img_path_cols, transforms=transform_list, num_workers=num_workers, include_test=test_df, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            
            if joint_training:
                domain_dataloaders = pool_domain_dataloaders(domain_dataloaders)

            print(f"\nTesting: {name} Ablation: {ablation}")

            if ablation == 'nomask':
                setup['ablation'] = 'focus'
            elif ablation == 'onlysoc':
                setup['ablation'] = 'scene'
            elif ablation == 'onlyenv':
                setup['ablation'] = 'focus'
            else:
                setup['ablation'] = False

            if scene_as_label:
                setup['scene'] = 'label'
            dropout_rate=0.3
            model = DualBranchModel(dropout_rate=dropout_rate, setup=setup, branch_norm=branch_norm)
            dual_model = model.to(device)
            trainable_params = [p for p in dual_model.parameters() if p.requires_grad]
            lr=1e-3
            optimizer = torch.optim.Adam(trainable_params, lr=lr)
            buffer = NaiveRehearsalBuffer(buffer_size=buffer_size) if use_buffer else None

            epochs=epochs
            exp_name = f"{name}_{fold=}_{buffer_size=}_{epochs=}_{seed=}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            scheduler=(get_linear_schedule_with_warmup, 0.05)
            refresh_optimiser=False

            dualbranch_kwargs = {
                    'mse_criterion': loss_fn #nn.MSELoss(),
                }

            config = {
                "timestamp": datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
                "model": {
                    "name": str(type(model)),
                    "setup": setup,
                    "branch_norm":branch_norm,
                    "dropout_rate":dropout_rate,
                },
                "buffer": str(type(buffer)),
                "buffer_size": buffer_size,
                "optimizer": str(type(optimizer)),
                "learning_rate": lr,
                "scheduler": {
                    "type": "transformers.get_linear_schedule_with_warmup",
                    "warmup": scheduler[1],
                },
                "device": str(device),
                "loss": str(dualbranch_kwargs['mse_criterion']),
                "joint_training": joint_training, #trained on pooled domains, not continual learning
                "seed": seed,
                "domains_order": domains,
                "train_val_df": train_val_path,
                "test_df": test_path,
                "hdf5_dataset_path": hdf5_dataset_path,
                "input_columns": img_path_cols,
                "input_transforms": [serialize_transform(x) for x in transform_list],
                "dataloader": {
                    "batch_size": batch,
                    "num_workers":num_workers,
                    "pin_memory": pin_memory,
                    "persistent_workers": persistent_workers,
                },
                "epochs": epochs,
                "early_stopping": {
                    "patience":15,
                    "delta":1e-3,
                },
                "gradient_clipping": grad_clipping,
                "refresh_optimiser": refresh_optimiser,
            }

            with open(f"../checkpoints/{exp_name}_config.json", "w") as f:
                json.dump(config, f, indent=4)

            unified_train_loop(
                model=dual_model,
                domains=domains if not joint_training else ['joint'],
                domain_dataloaders=domain_dataloaders,
                buffer=buffer,
                optimizer=optimizer,
                device=device,
                batch_fn=heuristic_dualbranch_batch,
                batch_kwargs=dualbranch_kwargs,
                num_epochs=epochs,
                exp_name=exp_name,
                gradient_clipping=grad_clipping,
                checkpoint_dir="../checkpoints",
                validation_set='val',
                scheduler=scheduler,
                refresh_optimiser=refresh_optimiser,
                early_stopping=earlystopping,
            )

In [ ]:
python main.py --experiment_id officemanners_dil_fold0 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --plot_results

In [ ]:
python main.py --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 3 --output_folder ./output --tensorboard --plot_results

In [ ]:
python main.py --experiment_id domainIL_exp_custom_hdf5_dn4il --seed 42 --model duca --dataset custom-hdf5-regression --buffer_size 120 --aux shape --lr 0.05 --n_epochs 50 --img_size 64 --shape_filter sobel --batch_size 8 --minibatch_size 8 --output_dir ./output/ --loss_wt 0.1 0.1 0.01 0.01 --ema_alpha 0.999 --ema_update_freq 0.06 --loss_type l2 --save_model --alpha_mm 0.1 0.1 --beta_mm 0.1 0.1

In [ ]:
python main.py --experiment_id officemanners_dil_fold0 --seed 42 --model duca --dataset custom-hdf5-regression --buffer_size 120 --aux shape --lr 0.05 --n_epochs 50 --img_size 64 --shape_filter sobel --batch_size 8 --minibatch_size 8 --output_dir ./output/ --loss_wt 0.1 0.1 0.01 0.01 --ema_alpha 0.999 --ema_update_freq 0.06 --loss_type l2 --save_model --alpha_mm 0.1 0.1 --beta_mm 0.1 0.1

In [ ]:
python main.py --experiment_id officemanners_dil_fold1 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results && python main.py --experiment_id officemanners_dil_fold2 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results && python main.py --experiment_id officemanners_dil_fold3 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results && python main.py --experiment_id officemanners_dil_fold4 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results

In [ ]:
# archive_domain_models('heuristicsplit.base_buffer_size=120_epochs=20_seed=42_20251014_171808', '../checkpoints')
python main.py --experiment_id officemanners_dil_fold0 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results